# Hyperparameter Sweep Analysis
## Joining and Visualizing Accuracy and Robustness Metrics

This notebook joins two hyperparameter sweep dataframes and generates comprehensive visualizations to identify important relationships and trends between hyperparameters and performance metrics.

## 1. Import Required Libraries

Import pandas, numpy, matplotlib, seaborn, and plotly for data manipulation and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load CSV Files

Read the two CSV files containing hyperparameter sweeps and corresponding accuracy/robustness measures.

In [ ]:
# Load the CSV files
# Update these paths to match your actual CSV file locations
df_accuracy = pd.read_csv('/Users/hannah/mit/vit_sae/accuracy_sweep.csv')
df_robustness = pd.read_csv('/Users/hannah/mit/vit_sae/robustness_sweep.csv')

print("Accuracy DataFrame loaded successfully")
print("Robustness DataFrame loaded successfully")

## 3. Inspect DataFrames

Display the first few rows, shape, data types, and summary statistics for both dataframes.

In [ ]:
print("=" * 80)
print("ACCURACY DATAFRAME")
print("=" * 80)
print(f"\nShape: {df_accuracy.shape}")
print(f"\nFirst few rows:")
print(df_accuracy.head())
print(f"\nData types:")
print(df_accuracy.dtypes)
print(f"\nSummary statistics:")
print(df_accuracy.describe())
print(f"\nColumn names: {df_accuracy.columns.tolist()}")

In [ ]:
print("\n" + "=" * 80)
print("ROBUSTNESS DATAFRAME")
print("=" * 80)
print(f"\nShape: {df_robustness.shape}")
print(f"\nFirst few rows:")
print(df_robustness.head())
print(f"\nData types:")
print(df_robustness.dtypes)
print(f"\nSummary statistics:")
print(df_robustness.describe())
print(f"\nColumn names: {df_robustness.columns.tolist()}")

## 4. Merge DataFrames on Hyperparameters

Join the two dataframes on the common hyperparameter columns (alpha, beta, lr, expansion, robustness).

In [ ]:
# Define the hyperparameter columns to join on
hyperparameter_cols = ['alpha', 'beta', 'lr', 'expansion', 'robustness']

# Check which columns exist in both dataframes
print("Columns in accuracy DataFrame:")
print(df_accuracy.columns.tolist())
print("\nColumns in robustness DataFrame:")
print(df_robustness.columns.tolist())

# Identify common hyperparameter columns that exist in both dataframes
common_cols = [col for col in hyperparameter_cols if col in df_accuracy.columns and col in df_robustness.columns]
print(f"\nCommon hyperparameter columns found: {common_cols}")

# Merge the dataframes on common hyperparameter columns
if len(common_cols) > 0:
    df_merged = pd.merge(df_accuracy, df_robustness, on=common_cols, how='inner')
    print(f"\nMerged DataFrame shape: {df_merged.shape}")
    print(f"\nFirst few rows of merged DataFrame:")
    print(df_merged.head())
else:
    print("Warning: No common hyperparameter columns found. Performing cross-join.")
    df_merged = df_accuracy.copy()
    # You may need to adjust this based on your actual data structure

## 5. Handle Missing Values

Identify and handle missing values using dropna(), fillna(), or other appropriate methods.

In [ ]:
print("Missing values before cleaning:")
print(df_merged.isnull().sum())
print(f"\nTotal missing values: {df_merged.isnull().sum().sum()}")

# Remove rows with missing values
df_merged_clean = df_merged.dropna()

print(f"\nShape before cleaning: {df_merged.shape}")
print(f"Shape after cleaning: {df_merged_clean.shape}")
print(f"Rows removed: {df_merged.shape[0] - df_merged_clean.shape[0]}")

print("\nMissing values after cleaning:")
print(df_merged_clean.isnull().sum())

In [ ]:
# Display the cleaned merged dataframe
print("\nCleaned Merged DataFrame:")
print(df_merged_clean.head(10))
print(f"\nFinal shape: {df_merged_clean.shape}")
print(f"\nData types:")
print(df_merged_clean.dtypes)

## 6. Create Scatter Plots for Hyperparameter Relationships

Generate scatter plots to visualize relationships between pairs of hyperparameters and their corresponding accuracy/robustness values.

In [ ]:
# Identify numeric columns (hyperparameters and metrics)
numeric_cols = df_merged_clean.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns available: {numeric_cols}")

# Get hyperparameters and performance metrics
hyperparams = [col for col in common_cols if col in df_merged_clean.columns]
metrics = [col for col in numeric_cols if col not in hyperparams]

print(f"\nHyperparameters: {hyperparams}")
print(f"Performance metrics: {metrics}")

In [ ]:
# Create scatter plots for hyperparameter relationships with metrics
if len(metrics) > 0 and len(hyperparams) > 0:
    fig, axes = plt.subplots(len(hyperparams), len(metrics), figsize=(15, 12))
    fig.suptitle('Hyperparameter vs Performance Metrics', fontsize=16, y=1.00)
    
    # Flatten axes for easier iteration if there's only one metric or hyperparameter
    if len(metrics) == 1 or len(hyperparams) == 1:
        axes = axes.flatten()
    
    plot_idx = 0
    for i, hyperparam in enumerate(hyperparams):
        for j, metric in enumerate(metrics):
            ax = axes[i * len(metrics) + j] if len(metrics) > 1 and len(hyperparams) > 1 else axes[plot_idx]
            
            ax.scatter(df_merged_clean[hyperparam], df_merged_clean[metric], alpha=0.6, s=50)
            ax.set_xlabel(hyperparam, fontsize=10)
            ax.set_ylabel(metric, fontsize=10)
            ax.set_title(f'{metric} vs {hyperparam}', fontsize=11)
            ax.grid(True, alpha=0.3)
            
            plot_idx += 1
    
    plt.tight_layout()
    plt.savefig('/Users/hannah/mit/vit_sae/scatter_plots_hyperparameters.png', dpi=300, bbox_inches='tight')
    print("Scatter plots saved to scatter_plots_hyperparameters.png")
    plt.show()
else:
    print("Insufficient data to create scatter plots")

## 7. Generate Heatmaps for Correlation Analysis

Create correlation heatmaps using seaborn to identify relationships between hyperparameters and performance metrics.

In [ ]:
# Calculate correlation matrix for numeric columns
correlation_matrix = df_merged_clean[numeric_cols].corr()

print("Correlation Matrix:")
print(correlation_matrix)

# Create correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Hyperparameters and Performance Metrics', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('/Users/hannah/mit/vit_sae/correlation_heatmap.png', dpi=300, bbox_inches='tight')
print("\nCorrelation heatmap saved to correlation_heatmap.png")
plt.show()

## 8. Create 3D Visualizations

Use plotly to create 3D scatter plots showing relationships between three hyperparameters and a performance metric.

In [ ]:
# Create 3D scatter plots
if len(hyperparams) >= 3 and len(metrics) > 0:
    # Select first three hyperparameters and first metric for visualization
    x_param = hyperparams[0]
    y_param = hyperparams[1]
    z_param = hyperparams[2]
    color_metric = metrics[0]
    
    fig = go.Figure(data=[go.Scatter3d(
        x=df_merged_clean[x_param],
        y=df_merged_clean[y_param],
        z=df_merged_clean[z_param],
        mode='markers',
        marker=dict(
            size=5,
            color=df_merged_clean[color_metric],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title=color_metric)
        ),
        text=[f'{x_param}: {x:.4f}<br>{y_param}: {y:.4f}<br>{z_param}: {z:.4f}<br>{color_metric}: {c:.4f}' 
              for x, y, z, c in zip(df_merged_clean[x_param], df_merged_clean[y_param], 
                                     df_merged_clean[z_param], df_merged_clean[color_metric])],
        hoverinfo='text'
    )])
    
    fig.update_layout(
        title=f'3D Visualization: {x_param}, {y_param}, {z_param} vs {color_metric}',
        scene=dict(
            xaxis_title=x_param,
            yaxis_title=y_param,
            zaxis_title=z_param
        ),
        width=900,
        height=800
    )
    
    fig.write_html('/Users/hannah/mit/vit_sae/3d_scatter_plot_1.html')
    print(f"3D scatter plot saved to 3d_scatter_plot_1.html")
    fig.show()
else:
    print("Insufficient hyperparameters for 3D visualization")

In [ ]:
# Create additional 3D plot with different metric if available
if len(hyperparams) >= 3 and len(metrics) > 1:
    x_param = hyperparams[0]
    y_param = hyperparams[1]
    z_param = hyperparams[2]
    color_metric = metrics[1]
    
    fig = go.Figure(data=[go.Scatter3d(
        x=df_merged_clean[x_param],
        y=df_merged_clean[y_param],
        z=df_merged_clean[z_param],
        mode='markers',
        marker=dict(
            size=5,
            color=df_merged_clean[color_metric],
            colorscale='Plasma',
            showscale=True,
            colorbar=dict(title=color_metric)
        ),
        text=[f'{x_param}: {x:.4f}<br>{y_param}: {y:.4f}<br>{z_param}: {z:.4f}<br>{color_metric}: {c:.4f}' 
              for x, y, z, c in zip(df_merged_clean[x_param], df_merged_clean[y_param], 
                                     df_merged_clean[z_param], df_merged_clean[color_metric])],
        hoverinfo='text'
    )])
    
    fig.update_layout(
        title=f'3D Visualization: {x_param}, {y_param}, {z_param} vs {color_metric}',
        scene=dict(
            xaxis_title=x_param,
            yaxis_title=y_param,
            zaxis_title=z_param
        ),
        width=900,
        height=800
    )
    
    fig.write_html('/Users/hannah/mit/vit_sae/3d_scatter_plot_2.html')
    print(f"3D scatter plot saved to 3d_scatter_plot_2.html")
    fig.show()

## 9. Analyze Performance Trends by Individual Hyperparameters

Create line plots and box plots to analyze how each hyperparameter individually affects accuracy and robustness.

In [ ]:
# Analyze trends for each hyperparameter
if len(hyperparams) > 0 and len(metrics) > 0:
    fig, axes = plt.subplots(len(hyperparams), 2, figsize=(16, 4 * len(hyperparams)))
    
    # Ensure axes is always 2D
    if len(hyperparams) == 1:
        axes = axes.reshape(1, -1)
    
    for idx, hyperparam in enumerate(hyperparams):
        # Sort by hyperparameter for line plot
        df_sorted = df_merged_clean.sort_values(by=hyperparam)
        
        # Line plot
        for metric in metrics:
            axes[idx, 0].plot(df_sorted[hyperparam], df_sorted[metric], marker='o', label=metric, alpha=0.7)
        
        axes[idx, 0].set_xlabel(hyperparam, fontsize=11)
        axes[idx, 0].set_ylabel('Metric Value', fontsize=11)
        axes[idx, 0].set_title(f'Trends: {hyperparam} vs Performance Metrics', fontsize=12)
        axes[idx, 0].legend()
        axes[idx, 0].grid(True, alpha=0.3)
        
        # Box plot
        df_merged_clean.boxplot(column=metrics, ax=axes[idx, 1])
        axes[idx, 1].set_title(f'Distribution of Metrics', fontsize=12)
        axes[idx, 1].set_ylabel('Metric Value', fontsize=11)
    
    plt.tight_layout()
    plt.savefig('/Users/hannah/mit/vit_sae/performance_trends.png', dpi=300, bbox_inches='tight')
    print("Performance trend plots saved to performance_trends.png")
    plt.show()

In [ ]:
# Create individual line plots for each hyperparameter and metric combination
if len(hyperparams) > 0 and len(metrics) > 0:
    for metric in metrics:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        
        for idx, hyperparam in enumerate(hyperparams[:4]):
            # Sort by hyperparameter
            df_sorted = df_merged_clean.sort_values(by=hyperparam)
            
            axes[idx].plot(df_sorted[hyperparam], df_sorted[metric], marker='o', linewidth=2, markersize=6)
            axes[idx].fill_between(range(len(df_sorted)), df_sorted[metric], alpha=0.3)
            axes[idx].set_xlabel(hyperparam, fontsize=11)
            axes[idx].set_ylabel(metric, fontsize=11)
            axes[idx].set_title(f'{metric} vs {hyperparam}', fontsize=12)
            axes[idx].grid(True, alpha=0.3)
        
        plt.suptitle(f'Impact of Hyperparameters on {metric}', fontsize=14, y=1.00)
        plt.tight_layout()
        plt.savefig(f'/Users/hannah/mit/vit_sae/trend_analysis_{metric}.png', dpi=300, bbox_inches='tight')
        print(f"Trend analysis plot for {metric} saved")
        plt.show()

## 10. Export Visualizations

Save all generated visualizations as PNG or HTML files for documentation and reporting.

In [ ]:
# Create a summary report
print("\n" + "=" * 80)
print("VISUALIZATION EXPORT SUMMARY")
print("=" * 80)

exported_files = [
    'scatter_plots_hyperparameters.png',
    'correlation_heatmap.png',
    '3d_scatter_plot_1.html',
    '3d_scatter_plot_2.html',
    'performance_trends.png',
]

print("\nExported visualizations:")
for i, filename in enumerate(exported_files, 1):
    filepath = f'/Users/hannah/mit/vit_sae/{filename}'
    print(f"{i}. {filename}")

print("\n" + "=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)
print(f"\nTotal merged records: {df_merged_clean.shape[0]}")
print(f"Hyperparameters analyzed: {', '.join(hyperparams)}")
print(f"Performance metrics: {', '.join(metrics)}")
print(f"\nCorrelation with metrics:")
for metric in metrics:
    print(f"\n{metric}:")
    correlations = correlation_matrix[metric].drop(metric).sort_values(ascending=False)
    print(correlations)

In [ ]:
# Create an interactive dashboard with plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=tuple([f"{h} vs {metrics[0]}" for h in hyperparams[:4]]),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'scatter'}]]
)

for i, hyperparam in enumerate(hyperparams[:4]):
    row = (i // 2) + 1
    col = (i % 2) + 1
    
    fig.add_trace(
        go.Scatter(
            x=df_merged_clean[hyperparam],
            y=df_merged_clean[metrics[0]],
            mode='markers',
            name=hyperparam,
            marker=dict(size=8, opacity=0.7)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(title_text=hyperparam, row=row, col=col)
    fig.update_yaxes(title_text=metrics[0], row=row, col=col)

fig.update_layout(
    title_text="Interactive Hyperparameter Analysis Dashboard",
    height=800,
    width=1200,
    showlegend=True
)

fig.write_html('/Users/hannah/mit/vit_sae/interactive_dashboard.html')
print("Interactive dashboard saved to interactive_dashboard.html")
fig.show()

In [ ]:
# Generate key insights
print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

for metric in metrics:
    print(f"\n{metric.upper()}:")
    print(f"  Min: {df_merged_clean[metric].min():.4f}")
    print(f"  Max: {df_merged_clean[metric].max():.4f}")
    print(f"  Mean: {df_merged_clean[metric].mean():.4f}")
    print(f"  Std Dev: {df_merged_clean[metric].std():.4f}")
    
    # Find best configuration
    best_idx = df_merged_clean[metric].idxmax()
    print(f"\n  Best configuration for {metric}:")
    for hyperparam in hyperparams:
        print(f"    {hyperparam}: {df_merged_clean.loc[best_idx, hyperparam]:.4f}")

In [ ]:
# Generate key insights
print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

for metric in metrics:
    print(f"\n{metric.upper()}:")
    print(f"  Min: {df_merged_clean[metric].min():.4f}")
    print(f"  Max: {df_merged_clean[metric].max():.4f}")
    print(f"  Mean: {df_merged_clean[metric].mean():.4f}")
    print(f"  Std Dev: {df_merged_clean[metric].std():.4f}")
    
    # Find best configuration
    best_idx = df_merged_clean[metric].idxmax()
    print(f"\n  Best configuration for {metric}:")
    for hyperparam in hyperparams:
        print(f"    {hyperparam}: {df_merged_clean.loc[best_idx, hyperparam]:.4f}")